# Phase 7: Binary Attack Detection Experiment
### Smart Grid Cyber-Physical Intrusion Detection

This notebook trains, evaluates, and compares standard and deep learning models for binary intrusion detection (`Normal` vs. `Attack`):
1. **Logistic Regression** (Linear baseline with class weighting)
2. **Decision Tree** (Interpretable tree baseline)
3. **Random Forest** (Ensemble bagging)
4. **XGBoost** (Gradient boosted trees)
5. **Support Vector Machine (SVM)** (Margin classifier with probability calibration)
6. **Deep Tabular MLP** (PyTorch Neural Network with BatchNorm, LeakyReLU, and Dropout)

**Target Metrics:** Accuracy, Attack Recall, Attack Precision, Macro F1, False Positive Rate (FPR), False Negative Rate (FNR), Inference Latency.


In [ ]:
# --- Colab / Local Setup ---
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score

if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists('../src'):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')
sys.path.insert(0, PROJECT_ROOT)

from src.data_loader import load_data4cyber, load_cicflowmeter_dnp3, get_preprocessor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on Device: {DEVICE}")


## 1. Load Dataset (`data4cyber_dataset` as Primary Cyber-Physical Benchmark)
Strictly cleans ground-truth attacker logs to eliminate target leakage.


In [ ]:
print("Loading data4cyber_dataset...")
data = load_data4cyber(random_state=42)

X_train, y_train = data['X_train'], data['y_train_binary']
X_val, y_val = data['X_val'], data['y_val_binary']
X_test, y_test = data['X_test'], data['y_test_binary']
features = data['feature_names']

print(f"Training samples: {len(X_train)} | Val samples: {len(X_val)} | Test samples: {len(X_test)}")
print(f"Feature count: {len(features)}")
print(f"Class counts in Test: {np.bincount(y_test)} (0: Benign, 1: Attack)")


## 2. Preprocessing Pipeline
Fitted **strictly on training data only** to prevent data snooping.


In [ ]:
prep_scaled = get_preprocessor(scaling=True)
prep_unscaled = get_preprocessor(scaling=False)

# Fit on Train only
X_tr_sc = prep_scaled.fit_transform(X_train)
X_va_sc = prep_scaled.transform(X_val)
X_te_sc = prep_scaled.transform(X_test)

X_tr_raw = prep_unscaled.fit_transform(X_train)
X_te_raw = prep_unscaled.transform(X_test)


## 3. Train Classical Machine Learning Models


In [ ]:
models = {
    "Logistic_Regression": (LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42), True),
    "Decision_Tree": (DecisionTreeClassifier(max_depth=15, class_weight='balanced', random_state=42), False),
    "Random_Forest": (RandomForestClassifier(n_estimators=100, max_depth=20, class_weight='balanced_subsample', n_jobs=-1, random_state=42), False),
    "XGBoost": (XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, n_jobs=-1, eval_metric='logloss', random_state=42), False),
    "SVM": (SGDClassifier(loss='modified_huber', max_iter=1000, class_weight='balanced', random_state=42), True)
}

results = []
trained_models = {}

for name, (clf, scaled) in models.items():
    print(f"Training {name}...")
    X_tr = X_tr_sc if scaled else X_tr_raw
    X_te = X_te_sc if scaled else X_te_raw
    
    t0 = time.time()
    clf.fit(X_tr, y_train)
    tr_time = time.time() - t0
    trained_models[name] = clf
    
    t0 = time.time()
    y_pred = clf.predict(X_te)
    inf_time_ms = ((time.time() - t0) / len(X_te)) * 1000.0
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    
    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Attack_Recall': round(rec, 4),
        'Attack_Precision': round(prec, 4),
        'Macro_F1': round(macro_f1, 4),
        'FPR': round(fpr, 4),
        'FNR': round(fnr, 4),
        'Latency_ms': round(inf_time_ms, 4),
        'Training_Time_s': round(tr_time, 2)
    })

df_res = pd.DataFrame(results)
display(df_res)


## 4. Train Deep Tabular Neural Network (PyTorch MLP)


In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, in_features):
        super(TabularMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

mlp = TabularMLP(X_tr_sc.shape[1]).to(DEVICE)
crit = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)

tx_tr = torch.tensor(X_tr_sc, dtype=torch.float32)
ty_tr = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(tx_tr, ty_tr), batch_size=128, shuffle=True)

print("Training PyTorch Tabular MLP...")
mlp.train()
for ep in range(15):
    for bx, by in loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        opt.zero_grad()
        loss = crit(mlp(bx), by)
        loss.backward()
        opt.step()

mlp.eval()
with torch.no_grad():
    tx_te = torch.tensor(X_te_sc, dtype=torch.float32).to(DEVICE)
    logits = mlp(tx_te)
    probs = torch.sigmoid(logits).cpu().numpy().flatten()
    mlp_preds = (probs >= 0.5).astype(int)

mlp_f1 = f1_score(y_test, mlp_preds, average='macro')
mlp_rec = recall_score(y_test, mlp_preds)
print(f"Deep MLP Macro F1: {mlp_f1:.4f} | Recall: {mlp_rec:.4f}")


## 5. Visualizing Confusion Matrices & Feature Importances


In [ ]:
plt.figure(figsize=(12, 4.5))

plt.subplot(1, 2, 1)
cm_xgb = confusion_matrix(y_test, trained_models['XGBoost'].predict(X_te_raw))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Attack'], yticklabels=['Benign', 'Attack'])
plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.subplot(1, 2, 2)
xgb = trained_models['XGBoost']
imp = pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=False).head(10)
imp.plot(kind='barh', color='#2b5c8f')
plt.title("Top 10 Most Important Features (XGBoost)")
plt.xlabel("Relative Importance")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()
